In [ ]:
# # laptop_csv_receiver.py
# import socket
# import json
# import csv
# from datetime import datetime

# sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
# sock.bind(("0.0.0.0", 8888))

# csv_file = open('gita_coordinates_live.csv', 'w', newline='')
# csv_writer = csv.writer(csv_file)
# csv_writer.writerow(['timestamp', 'gita_x', 'gita_y', 'gita_z', 'yaw', 'unity_x', 'unity_z', 'scale_factor'])

# print("Listening for VR data... CSV: gita_coordinates_live.csv")

# while True:
#     data, addr = sock.recvfrom(1024)
#     cmd = json.loads(data.decode('utf-8'))
    
#     csv_writer.writerow([
#         cmd['timestamp'],
#         cmd['x'],
#         cmd['y'],
#         cmd['z'],
#         cmd['yaw'],
#         cmd['unity_x'],
#         cmd['unity_z'],
#         cmd['scale_factor']
#     ])
#     csv_file.flush()  # Write immediately
    
#     print(f"Gita: ({cmd['x']:.2f}, {cmd['z']:.2f}) @ {cmd['yaw']:.1f}°")

In [33]:
#!/usr/bin/env python3
"""
Relay script for Mac (Laptop 1)
Receives data from Quest/Unity → Forwards to Laptop 2
"""

import socket
import sys
import csv
import os
from datetime import datetime

# Configuration
RECEIVE_PORT = 8888
LAPTOP2_IP = "172.168.1.14"  
LAPTOP2_PORT = 8888
CSV_FILE = f"relay_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

# Setup sockets
receive_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
receive_sock.bind(("0.0.0.0", RECEIVE_PORT))

send_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
laptop2_endpoint = (LAPTOP2_IP, LAPTOP2_PORT)

# Setup CSV
csv_file = open(CSV_FILE, "w", newline="")
csv_writer = csv.writer(csv_file)
csv_writer.writerow([
    "relay_timestamp",
    "packet_num",
    "source_ip",
    "source_port",
    "dest_ip",
    "dest_port",
    "bytes",
    # Unity data fields
    "x",
    "y",
    "z",
    "yaw",
    "unity_timestamp",
    "unity_x",
    "unity_z",
    "scale_factor"
])

print("=" * 60)
print("MAC RELAY - VR → Laptop 2")
print("=" * 60)
print(f"Listening on:  0.0.0.0:{RECEIVE_PORT} (Quest/Unity)")
print(f"Forwarding to: {LAPTOP2_IP}:{LAPTOP2_PORT} (Laptop 2)")
print(f"Logging to:    {CSV_FILE}")
print("=" * 60)
print("\nWaiting for VR data...\n")

packet_count = 0

try:
    while True:
        data, addr = receive_sock.recvfrom(1024)
        packet_count += 1
        relay_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

        # Forward to Laptop 2
        send_sock.sendto(data, laptop2_endpoint)

        # Parse Unity CSV fields
        try:
            decoded = data.decode("utf-8", errors="replace").strip()
            fields = decoded.split(",")
            x, y, z, yaw, unity_ts, unity_x, unity_z, scale_factor = fields[:8]
        except (ValueError, IndexError):
            # If parsing fails, fill with raw data and blanks
            x = y = z = yaw = unity_ts = unity_x = unity_z = scale_factor = ""
            print(f"  ⚠ Packet {packet_count} could not be parsed: {data[:60]}")

        # Write to CSV
        csv_writer.writerow([
            relay_timestamp,
            packet_count,
            addr[0],
            addr[1],
            LAPTOP2_IP,
            LAPTOP2_PORT,
            len(data),
            x, y, z, yaw, unity_ts, unity_x, unity_z, scale_factor
        ])
        csv_file.flush()

        # Print status every 10 packets
        if packet_count % 10 == 0:
            print(f"✓ Relayed {packet_count} packets | From: {addr[0]} → To: {LAPTOP2_IP}")

except KeyboardInterrupt:
    print(f"\n\n✓ Relay stopped. Total packets: {packet_count}")
    print(f"✓ Log saved to: {os.path.abspath(CSV_FILE)}")
    csv_file.close()
    sys.exit(0)

MAC RELAY - VR → Laptop 2
Listening on:  0.0.0.0:8888 (Quest/Unity)
Forwarding to: 172.168.1.14:8888 (Laptop 2)
Logging to:    relay_log_20260311_210523.csv

Waiting for VR data...

✓ Relayed 10 packets | From: 172.168.1.2 → To: 172.168.1.14
✓ Relayed 20 packets | From: 172.168.1.2 → To: 172.168.1.14
✓ Relayed 30 packets | From: 172.168.1.2 → To: 172.168.1.14
✓ Relayed 40 packets | From: 172.168.1.2 → To: 172.168.1.14
✓ Relayed 50 packets | From: 172.168.1.2 → To: 172.168.1.14
✓ Relayed 60 packets | From: 172.168.1.2 → To: 172.168.1.14
✓ Relayed 70 packets | From: 172.168.1.2 → To: 172.168.1.14
✓ Relayed 80 packets | From: 172.168.1.2 → To: 172.168.1.14
✓ Relayed 90 packets | From: 172.168.1.2 → To: 172.168.1.14
✓ Relayed 100 packets | From: 172.168.1.2 → To: 172.168.1.14
✓ Relayed 110 packets | From: 172.168.1.2 → To: 172.168.1.14
✓ Relayed 120 packets | From: 172.168.1.2 → To: 172.168.1.14
✓ Relayed 130 packets | From: 172.168.1.2 → To: 172.168.1.14
✓ Relayed 140 packets | From: 172.

SystemExit: 0

In [ ]:
#!/usr/bin/env python3
"""
Relay script for Mac (Laptop 1)
Receives data from Quest/Unity → Forwards to Laptop 2
"""

import socket
import sys

# Configuration
RECEIVE_PORT = 8888          # Port to receive from Quest/Unity
LAPTOP2_IP = "10.30.148.126"   # Laptop 2's IP address
LAPTOP2_PORT = 8888          # Port to send to Laptop 2

# Setup sockets
receive_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
receive_sock.bind(("0.0.0.0", RECEIVE_PORT))

send_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
laptop2_endpoint = (LAPTOP2_IP, LAPTOP2_PORT)

print("=" * 60)
print("MAC RELAY - VR → Laptop 2")
print("=" * 60)
print(f"Listening on:  0.0.0.0:{RECEIVE_PORT} (Quest/Unity)")
print(f"Forwarding to: {LAPTOP2_IP}:{LAPTOP2_PORT} (Laptop 2)")
print("=" * 60)
print("\nWaiting for VR data...\n")

packet_count = 0

try:
    while True:
        # Receive from Quest/Unity
        data, addr = receive_sock.recvfrom(1024)
        packet_count += 1
        
        # Forward to Laptop 2
        send_sock.sendto(data, laptop2_endpoint)
        
        # Print status every 10 packets
        if packet_count % 10 == 0:
            print(f"✓ Relayed {packet_count} packets | From: {addr[0]} → To: {LAPTOP2_IP}")
        
except KeyboardInterrupt:
    print(f"\n\n✓ Relay stopped. Total packets: {packet_count}")
    sys.exit(0)


In [ ]:
# test_mac_receiver.py
import socket

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.bind(("0.0.0.0", 8888))

print("=" * 50)
print("Listening on Mac at 0.0.0.0:8888")
print("Waiting for ANY data from Quest...")
print("=" * 50)

while True:
    data, addr = sock.recvfrom(1024)
    print(f"\n✓✓✓ RECEIVED DATA ✓✓✓")
    print(f"From: {addr[0]}:{addr[1]}")
    print(f"Data: {data.decode('utf-8')[:200]}")
    print("=" * 50)